# Eda службы поддержки

Ноутбук показывает базовый анализ обращений: качество данных, sla, backlog, нагрузку агентов и категории с повышенным риском.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path('..')
processed = root / 'data' / 'processed'
marts = root / 'data' / 'marts'

In [ ]:
tickets = pd.read_csv(processed / 'tickets.csv', parse_dates=['created_at', 'first_response_at', 'closed_at'])
agents = pd.read_csv(processed / 'agents.csv')
departments = pd.read_csv(processed / 'departments.csv')
quality = pd.read_csv(processed / 'data_quality_summary.csv')
tickets.head()

## Проверка качества

Сначала смотрим автоматические проверки и базовые пропуски.

In [ ]:
quality

In [ ]:
tickets.isna().mean().sort_values(ascending=False).head(10)

## Объем обращений

Проверяем недельный ритм и всплески нагрузки.

In [ ]:
daily_volume = tickets.assign(date=tickets['created_at'].dt.date).groupby('date').size()
daily_volume.plot(figsize=(12, 4), title='дневной объем обращений')
plt.ylabel('tickets')
plt.show()

## Sla

Смотрим долю нарушений по приоритетам и категориям.

In [ ]:
sla_by_priority = tickets.groupby('priority')['sla_breached'].agg(['count', 'mean']).sort_values('mean', ascending=False)
sla_by_priority

In [ ]:
sla_by_category = tickets.groupby('category')['sla_breached'].agg(['count', 'mean']).sort_values('mean', ascending=False)
sla_by_category.head(10)

## Backlog

Используем готовую витрину, чтобы оценить рост открытых обращений.

In [ ]:
backlog = pd.read_csv(marts / 'mart_backlog.csv', parse_dates=['date'])
backlog.plot(x='date', y=['backlog_open_tickets', 'high_priority_backlog'], figsize=(12, 4))
plt.title('динамика backlog')
plt.show()

## Нагрузка агентов

Проверяем, как распределяется работа между агентами и где виден перегруз.

In [ ]:
workload = pd.read_csv(marts / 'mart_workload.csv')
agent_summary = workload.groupby(['agent_name', 'is_overloaded_group']).agg(
    assigned_tickets=('assigned_tickets', 'sum'),
    open_backlog=('open_backlog', 'mean'),
    sla_breach_rate=('sla_breach_rate', 'mean'),
    avg_satisfaction_score=('avg_satisfaction_score', 'mean'),
).sort_values('assigned_tickets', ascending=False)
agent_summary.head(12)

## Ключевые выводы

- Недельная сезонность задает основной ритм нагрузки.
- Категории с высокой сложностью чаще нарушают sla.
- Перегруженные агенты получают больше обращений и могут накапливать backlog.
- Удовлетворенность чувствительна к долгому решению и нарушению sla.